In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# =============================================================================
# 🏆 VISTA CODEFEST'26 — PLATINUM PIPELINE (INFRASTRUCTURE-GRADE ML)
# =============================================================================
# FEATURES:
#  - GPU enforced (fail-fast)
#  - Dynamic disk guard
#  - Category remapping (label-safe)
#  - Stratified difficulty split
#  - Hybrid storage (symlink train / copy val)
#  - YOLO worker safety mode
#  - Resume training support
#  - Checkpoint integrity validation
#  - Chunked inference (zero OOM risk)
#  - Validation alignment guarantee
#  - Judge-grade submission validation
#  - Auto-backup to /kaggle/temp
# =============================================================================

# -----------------------------------------------------------------------------
# FORCE GPU
# -----------------------------------------------------------------------------
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# -----------------------------------------------------------------------------
# IMPORTS
# -----------------------------------------------------------------------------
import subprocess
import sys
import json
import time
import random
import gc
import shutil
from pathlib import Path
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# GPU VERIFICATION
# -----------------------------------------------------------------------------
print("=" * 70)
print("🔍 GPU VERIFICATION")
print("=" * 70)

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        for line in result.stdout.split("\n"):
            if "Tesla" in line or "GPU" in line:
                print(line.strip())
except:
    print("⚠️ nvidia-smi unavailable")

# -----------------------------------------------------------------------------
# DEPENDENCIES
# -----------------------------------------------------------------------------
try:
    import ultralytics
except ImportError:
    print("📦 Installing ultralytics...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

import yaml
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# -----------------------------------------------------------------------------
# GPU FAIL-FAST
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("🎯 GPU CHECK")
print("=" * 70)

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU REQUIRED — Enable P100 in Kaggle")

print(f"✅ CUDA: {torch.version.cuda}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

t = torch.zeros(1).cuda()
print(f"✅ GPU Tensor Test: {t.device}")
del t

DEVICE = 0
print("=" * 70)

# -----------------------------------------------------------------------------
# DISK GUARD (DYNAMIC)
# -----------------------------------------------------------------------------
print("\n💾 DISK CHECK")
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Free space: {free/1e9:.1f} GB")

REQUIRED_GB = max(12, free / 1e9 * 0.4)  # adaptive threshold
if free < REQUIRED_GB * 1e9:
    raise RuntimeError(f"❌ Disk too low: {free/1e9:.1f}GB free, need {REQUIRED_GB:.1f}GB")

# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
class Cfg:
    ROOT = "/kaggle/input/vista26"
    BASE = f"{ROOT}/Vistas Dataset Public/Vistas Dataset Public"
    WORK = "/kaggle/working"

    TRAIN_DIR = f"{BASE}/train"
    TEST_DIR  = f"{BASE}/test"
    VAL_DIR   = f"{BASE}/validation"

    TRAIN_JSON = f"{BASE}/instances_train.json"
    TEST_JSON  = f"{BASE}/instances_test.json"
    VAL_JSON   = f"{ROOT}/instances_val.json"
    CATS_JSON  = f"{BASE}/Categories.json"

    YOLO_MODEL = "yolov8n.pt"
    YOLO_EPOCHS = 20
    YOLO_IMGSZ = 640
    YOLO_BATCH = 24
    YOLO_WORKERS = 0  # CRITICAL: symlink-safe
    YOLO_PATIENCE = 5

    TRAIN_SAMPLE_RATIO = 0.3
    VAL_RATIO = 0.10

    CONF = 0.25
    IOU = 0.45
    MAX_DET = 50

    SEED = 42

C = Cfg()
random.seed(C.SEED)
np.random.seed(C.SEED)
torch.manual_seed(C.SEED)
torch.cuda.manual_seed_all(C.SEED)

START = time.time()
def elapsed():
    return f"{(time.time()-START)/60:.1f} min"

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -----------------------------------------------------------------------------
# AUTO RESUME SUPPORT
# -----------------------------------------------------------------------------
RESUME_PT = f"{C.WORK}/yolo_run/weights/last.pt"
if os.path.exists(RESUME_PT):
    print("♻️ Resuming from checkpoint")
    C.YOLO_MODEL = RESUME_PT

# -----------------------------------------------------------------------------
# STAGE 0 — LOAD DATA
# -----------------------------------------------------------------------------
print(f"\n[{elapsed()}] STAGE 0 — Loading Data")

with open(C.CATS_JSON) as f:
    cats_raw = json.load(f)["categories"]

cats_raw.sort(key=lambda c: int(c["id"]))
VALID_IDS = set(int(c["id"]) for c in cats_raw)

cat_id_to_yolo = {int(c["id"]): i for i, c in enumerate(cats_raw)}
names_map = {i: c["name"] for i, c in enumerate(cats_raw)}

def load_data(json_path):
    with open(json_path) as f:
        return json.load(f)

train_data = load_data(C.TRAIN_JSON)
test_data = load_data(C.TEST_JSON)

def parse_images(data):
    imgs = {}
    for img in data["images"]:
        imgs[int(img["id"])] = {
            "file_name": img["file_name"],
            "width": int(img["width"]),
            "height": int(img["height"]),
            "level": img.get("level", "unknown")
        }
    return imgs

def parse_annotations(data, valid_ids):
    anns = defaultdict(list)
    for ann in data["annotations"]:
        iid = int(ann["image_id"])
        cid = int(ann["category_id"])
        if cid in valid_ids:
            anns[iid].append({
                "category_id": cid,
                "bbox": ann["bbox"]
            })
    return anns

train_img_info = parse_images(train_data)
test_img_info = parse_images(test_data)

train_ann_by_img = parse_annotations(train_data, VALID_IDS)
test_ann_by_img = parse_annotations(test_data, VALID_IDS)

train_ids = [iid for iid in train_img_info if train_ann_by_img[iid]]
test_ids = [iid for iid in test_img_info if test_ann_by_img[iid]]

random.shuffle(train_ids)
train_ids = train_ids[:int(len(train_ids) * C.TRAIN_SAMPLE_RATIO)]

# -----------------------------------------------------------------------------
# STRATIFIED SPLIT
# -----------------------------------------------------------------------------
by_level = defaultdict(list)
for iid in test_ids:
    by_level[test_img_info[iid]["level"]].append(iid)

train_multi, val_ids = [], []
for lvl in ["easy", "medium", "hard"]:
    ids = by_level[lvl]
    random.shuffle(ids)
    split = int(len(ids) * (1 - C.VAL_RATIO))
    train_multi.extend(ids[:split])
    val_ids.extend(ids[split:])

# -----------------------------------------------------------------------------
# DATASET CREATION
# -----------------------------------------------------------------------------
YOLO_DIR = f"{C.WORK}/yolo_data"
shutil.rmtree(YOLO_DIR, ignore_errors=True)

for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{YOLO_DIR}/{sub}", exist_ok=True)

def write_yolo(ids, split, img_info, ann_info, img_dir):
    written = skipped = 0
    for iid in ids:
        info = img_info[iid]
        src = f"{img_dir}/{info['file_name']}"
        if not os.path.exists(src):
            skipped += 1
            continue

        stem = f"{iid}_{Path(info['file_name']).stem}"
        img_dst = f"{YOLO_DIR}/images/{split}/{stem}.jpg"
        lbl_dst = f"{YOLO_DIR}/labels/{split}/{stem}.txt"

        try:
            if split == "train":
                os.symlink(src, img_dst)
                if not os.path.exists(img_dst):
                    raise OSError()
            else:
                shutil.copy2(src, img_dst)
        except:
            shutil.copy2(src, img_dst)

        W, H = info["width"], info["height"]
        lines = []
        for ann in ann_info[iid]:
            x, y, w, h = ann["bbox"]
            cls = cat_id_to_yolo[ann["category_id"]]
            cx = (x + w/2) / W
            cy = (y + h/2) / H
            nw = w / W
            nh = h / H
            lines.append(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        with open(lbl_dst, "w") as f:
            f.write("\n".join(lines))

        written += 1

    print(f"{split.upper()} → Written: {written:,} | Skipped: {skipped}")

write_yolo(train_ids, "train", train_img_info, train_ann_by_img, C.TRAIN_DIR)
write_yolo(train_multi, "train", test_img_info, test_ann_by_img, C.TEST_DIR)
write_yolo(val_ids, "val", test_img_info, test_ann_by_img, C.TEST_DIR)

# -----------------------------------------------------------------------------
# DATASET YAML
# -----------------------------------------------------------------------------
ds_yaml = {
    "path": C.WORK,
    "train": "yolo_data/images/train",
    "val": "yolo_data/images/val",
    "nc": len(names_map),
    "names": names_map
}

with open(f"{C.WORK}/dataset.yaml", "w") as f:
    yaml.dump(ds_yaml, f)

cleanup()

# -----------------------------------------------------------------------------
# STAGE 1 — TRAIN
# -----------------------------------------------------------------------------
print(f"\n[{elapsed()}] STAGE 1 — TRAINING")

model = YOLO(C.YOLO_MODEL)
model.train(
    data=f"{C.WORK}/dataset.yaml",
    epochs=C.YOLO_EPOCHS,
    imgsz=C.YOLO_IMGSZ,
    batch=C.YOLO_BATCH,
    device=DEVICE,
    workers=C.YOLO_WORKERS,
    project=C.WORK,
    name="yolo_run",
    exist_ok=True,
    patience=C.YOLO_PATIENCE,
    amp=True,
    cache=False,
    mosaic=0.5,
    mixup=0.0,
)

BEST_PT = f"{C.WORK}/yolo_run/weights/best.pt"
if not os.path.exists(BEST_PT):
    raise RuntimeError("❌ Best model not found")

cleanup()

# -----------------------------------------------------------------------------
# STAGE 2 — THRESHOLD SWEEP (CHUNKED)
# -----------------------------------------------------------------------------
print(f"\n[{elapsed()}] STAGE 2 — THRESHOLD SWEEP")

model = YOLO(BEST_PT)

detections = {}
BATCH = 32

val_yolo_paths = [
    f"{YOLO_DIR}/images/val/{iid}_{Path(test_img_info[iid]['file_name']).stem}.jpg"
    for iid in val_ids
]

for i in range(0, len(val_yolo_paths), BATCH):
    chunk = val_yolo_paths[i:i+BATCH]
    results = model.predict(chunk, conf=0.01, iou=C.IOU, max_det=C.MAX_DET, device=DEVICE, verbose=False)
    for iid, res in zip(val_ids[i:i+BATCH], results):
        if res.boxes is None:
            detections[iid] = []
        else:
            detections[iid] = list(zip(
                res.boxes.conf.cpu().tolist(),
                res.boxes.cls.cpu().int().tolist()
            ))

gt_counts = {iid: len(test_ann_by_img[iid]) for iid in val_ids}
gt_cats = {iid: [a["category_id"] for a in test_ann_by_img[iid]] for iid in val_ids}

best_conf, best_score = 0.25, 0.0
for conf in np.arange(0.05, 0.65, 0.01):
    score = 0.0
    for iid in val_ids:
        dets = [cls+1 for c, cls in detections[iid] if c >= conf]
        if len(dets) == gt_counts[iid]:
            inter = sum((Counter(dets) & Counter(gt_cats[iid])).values())
            score += inter / max(1, len(gt_cats[iid]))
    avg = score / len(val_ids)
    if avg > best_score:
        best_score = avg
        best_conf = round(float(conf), 3)

C.CONF = best_conf
print(f"Optimal conf = {C.CONF} | Score = {best_score:.4f}")

cleanup()

# -----------------------------------------------------------------------------
# STAGE 3 — FINAL INFERENCE
# -----------------------------------------------------------------------------
print(f"\n[{elapsed()}] STAGE 3 — FINAL INFERENCE")

with open(C.VAL_JSON) as f:
    val_meta = json.load(f)["images"]

val_targets = {int(i["id"]): f"{C.VAL_DIR}/{i['file_name']}" for i in val_meta}
val_ids_sorted = sorted(val_targets)

results_map = {}
for i in range(0, len(val_ids_sorted), BATCH):
    ids_chunk = val_ids_sorted[i:i+BATCH]
    paths = [val_targets[iid] for iid in ids_chunk]
    preds = model.predict(paths, conf=C.CONF, iou=C.IOU, max_det=C.MAX_DET, device=DEVICE, verbose=False)

    for iid, res in zip(ids_chunk, preds):
        if res.boxes is None:
            results_map[iid] = []
        else:
            results_map[iid] = sorted([int(cls)+1 for cls in res.boxes.cls.cpu().tolist()])

for iid in val_ids_sorted:
    results_map.setdefault(iid, [])

# -----------------------------------------------------------------------------
# STAGE 4 — SUBMISSION + VALIDATION
# -----------------------------------------------------------------------------
print(f"\n[{elapsed()}] STAGE 4 — SUBMISSION")

rows = []
for iid in val_ids_sorted:
    cats = [c for c in results_map[iid] if c in VALID_IDS]
    rows.append({"image_id": iid, "categories": json.dumps(sorted(cats))})

df = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)

assert df["image_id"].is_unique
assert len(df) == len(val_ids_sorted)
assert set(df["image_id"]) == set(val_ids_sorted)

for r in df["categories"]:
    lst = json.loads(r)
    assert lst == sorted(lst)
    for c in lst:
        assert c in VALID_IDS

OUT = f"{C.WORK}/submission.csv"
df.to_csv(OUT, index=False)
shutil.copy2(OUT, "/kaggle/temp/submission_backup.csv")

# -----------------------------------------------------------------------------
# STATS
# -----------------------------------------------------------------------------
total_objs = sum(len(json.loads(r)) for r in df["categories"])
empty = sum(1 for r in df["categories"] if r == "[]")

print(f"""
{'='*70}
🏆 PLATINUM PIPELINE COMPLETE
{'='*70}
File: {OUT}
Backup: /kaggle/temp/submission_backup.csv
Images: {len(df)}
Objects: {total_objs}
Empty: {empty}
Conf: {C.CONF}
Val Score: {best_score:.4f}
GPU: {torch.cuda.get_device_name(0)}
Time: {elapsed()}
{'='*70}

✅ Upload submission.csv to Kaggle
""")

🔍 GPU VERIFICATION
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
📦 Installing ultralytics...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🎯 GPU CHECK
✅ CUDA: 12.6
✅ GPU: Tesla P100-PCIE-16GB
✅ VRAM: 17.06 GB
✅ GPU Tensor Test: cuda:0

💾 DISK CHECK
Free space: 20.9 GB

[0.0 min] STAGE 0 — Loading Data
TRAIN → Written: 16,121 | Skippe

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/temp/submission_backup.csv'